# K-Nearest-Neighbor (KNN) — 2D Classification Walkthrough

A self-contained notebook version of this repository's from-scratch **K-Nearest Neighbors (KNN)** implementation, built to teach the two core ideas behind KNN:

1. **Distance comparison** — measuring how "close" a new point is to every known point.
2. **Majority vote** — letting the closest points "vote" on what class the new point belongs to.

No `scikit-learn`, no black box — every step is plain NumPy so you can read and trace exactly what the algorithm does.

This notebook mirrors the same workflow as `main.py` in this project, but everything (data, algorithm, plotting) is defined directly in the cells below instead of being spread across `Process/*.py` and `config_dev.yaml` — so it can be read and run top to bottom on its own. The original config-driven scripts (`main.py`, `config_dev.yaml`, `Process/`) are untouched and still work the same way as before; this notebook is an additional, standalone walkthrough.

**Requirements:** `numpy`, `matplotlib`, and `pandas` (all listed in `requirements.txt`).

## 1. What is KNN?

K-Nearest Neighbors is a **classification algorithm** based on a simple idea:

> "You are the average of the people (points) closest to you."

Given a new, unlabeled point, KNN:

1. Measures the distance from that point to **every** point in the training data.
2. Picks the **k** points with the smallest distance (the "nearest neighbors").
3. Looks at the **class label** of those k neighbors and predicts the label that appears **most often** (majority vote).

KNN doesn't "train" a model in the usual sense — there's no equation being fitted. It just remembers all the training data and does the distance + vote calculation at prediction time. This is called a **lazy learner** / **instance-based learner**.

## 2. The Dataset

Every data point here has:

| Field | Meaning |
| --- | --- |
| `x` | Hours streamed per week (a feature) |
| `y` | Hours spent in ranked/competitive play per week (a feature) |
| `label` | The class the account belongs to (`0`, `1`, `2`) |

**The scenario:** a game-streaming platform wants to auto-tag creator accounts into three groups based on how they actually use the platform, so it can route them to the right creator program:

- **Streamer (0)** — streams a lot, but treats it as entertainment/variety content rather than ranked competition (high streamed hours, low competing hours).
- **Gamer (1)** — a casual player: streams little to occasionally, plays a moderate amount, not seriously ranked (low-to-moderate on both axes).
- **E-Sports (2)** — a competitive/professional player: spends most of their time in ranked play, streams only occasionally (e.g. practice VODs) (low-to-moderate streamed hours, high competing hours).

Unlike the original 4-point toy example, this is a hand-authored table of 18 accounts (6 per class) with believable hour ranges, so the clusters and the decision boundary below actually look like real usage data instead of four arbitrary dots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle

# x = hours streamed per week, y = hours spent in ranked/competitive play per week
data_x = np.array([
    28, 22, 30, 18, 25, 33,      # Streamer
    6, 4, 9, 3, 8, 10,           # Gamer
    10, 5, 12, 7, 14, 4, 8         # E-Sports
], dtype=float)

data_y = np.array([
    4, 2.5, 6, 3, 1.5, 5,        # Streamer
    7, 5, 9, 4.5, 6, 10,         # Gamer
    22, 18, 26, 20, 28, 16.5, 32    # E-Sports
], dtype=float)

data_label = np.array([
    0, 0, 0, 0, 0, 0,
    1, 1, 1, 1, 1, 1,
    2, 2, 2, 2, 2, 2, 2,
])

label_names = {0: "Streamer", 1: "Gamer", 2: "E-Sports"}
colors = ["red", "blue", "green"]

xlabel = "Hours Streamed per Week"
ylabel = "Hours Competing (Ranked Play) per Week"

assert len(data_x) == len(data_y) == len(data_label) == 19
print(f"{len(data_x)} accounts loaded.")

### The `Data` container

A small container class for the dataset — same role as `Process/Data.py` in the original project — holding the coordinates, labels, and the colormap used for plotting. We'll also build a small pandas `DataFrame` right after it so the dataset is easy to inspect as a table.

In [ ]:
class Data:
    def __init__(self, data_x, data_y, data_label, colors=colors):
        self.x = data_x
        self.y = data_y
        self.label = data_label            # <-- group 0/1/2, separate from x/y
        self.color_list = colors
        self.cmap = ListedColormap(colors)

    def __len__(self):
        return len(self.x)


data = Data(data_x, data_y, data_label)

# View the dataset as a table instead of one print() per point
df = pd.DataFrame({
    "x": data.x,
    "y": data.y,
    "label": data.label,
    "label_name": [label_names[lbl] for lbl in data.label],
})
df.index.name = "point"

print(f"{len(data)} points total")
df

### Plotting the raw data

Before any prediction happens, let's just look at what the data looks like — a scatter plot colored by class. This is the notebook equivalent of `Plot.plot_data()`.

In [ ]:
def plot_data(data, xlabel, ylabel):
    plt.figure(figsize=(8, 6))
    plt.scatter(data.x, data.y, c=data.label, cmap=data.cmap, edgecolor='k')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title('Streaming Platform Accounts (3 Groups)')
    plt.colorbar(ticks=sorted(set(data.label)), label='Group')
    plt.show()


plot_data(data, xlabel=xlabel, ylabel=ylabel)

## 3. Step-by-Step: How the Algorithm Works

We'll build up the algorithm in three steps, the same way `Process/Calculate.py` does it, and demo each step against a single test account: someone who streams **9 hours/week** and plays ranked **13 hours/week** — a borderline profile that could plausibly be a very active Gamer or a part-time E-Sports player.

### Step 1 — Calculate distance

For a new point `(point_x, point_y)`, compute the **Euclidean distance** to every training point:

```text
distance = √( (x_i - point_x)² + (y_i - point_y)² )
```

In [ ]:
def calculate_distance(data_x, data_y, point_x, point_y):
    """Euclidean distance from (point_x, point_y) to every training point."""
    return np.sqrt((data_x - point_x) ** 2 + (data_y - point_y) ** 2)


test_point_x, test_point_y = 9, 13
distances = calculate_distance(data_x, data_y, test_point_x, test_point_y)
df_distances = pd.DataFrame({
    "point": range(len(data_x)),
    "x": data_x,
    "y": data_y,
    "label": data_label,
    "distance": distances
})
print(f"Distance from the test point to each of the {len(data_x)} accounts:")
print(df_distances)
print(f"Distance from test point sorted by distance:")
print(df_distances.sort_values("distance"))

Thanks to NumPy vectorization, this line calculates the distance to **all** points at once (no `for` loop needed).

### Step 2 — Find the k nearest neighbors

In [ ]:
def get_k_nearest_neighbors(data_x, data_y, data_label, point_x, point_y, k):
    """Labels of the k training points closest to (point_x, point_y)."""
    distances = calculate_distance(data_x, data_y, point_x, point_y)
    nearest_indices = np.argsort(distances)[:k]
    return data_label[nearest_indices]


k = 5
neighbor_labels = get_k_nearest_neighbors(data_x, data_y, data_label, test_point_x, test_point_y, k)
print(f"Labels of the {k} nearest neighbors:", [label_names[lbl] for lbl in neighbor_labels])

- `np.argsort(distances)` sorts the **indices** of the points by distance, smallest first.
- `[:k]` keeps only the first `k` — i.e. the indices of the `k` closest points.
- Looking those indices up in `data_label` gives the class of each of those neighbors.

### Step 3 — Majority vote

In [ ]:
def predict_numeric_label(data_x, data_y, data_label, point_x, point_y, k):
    """Majority-vote label (as a number) among the k nearest neighbors."""
    neighbors_labels = get_k_nearest_neighbors(data_x, data_y, data_label, point_x, point_y, k)
    unique_labels, counts = np.unique(neighbors_labels, return_counts=True)
    majority_label = unique_labels[np.argmax(counts)]
    return majority_label


winner = predict_numeric_label(data_x, data_y, data_label, test_point_x, test_point_y, k)
print(f"Majority vote result: label {winner} -> {label_names[winner]}")

- `np.unique(..., return_counts=True)` counts how many times each label shows up among the k neighbors.
- `np.argmax(counts)` finds which label has the **highest count** — the "winner" of the vote.

> ℹ️ Tie-breaking: if two labels get the same number of votes, `np.unique` returns labels sorted from smallest to largest, so `np.argmax` picks the **first** (numerically smallest) label among the tied ones.

### Putting it together: the `KNNCalculate` class

Now let's wrap these three steps into a reusable class — exactly what `Process/Calculate.py` provides, and what `main.py` actually uses. This is the version we'll use for the rest of the notebook.

In [ ]:
class KNNCalculate:
    def __init__(self, data_x, data_y, data_label, label_names=None):
        """
        label_names: dict mapping numeric label -> class name, e.g. {0: "Streamer", 1: "Gamer", 2: "E-Sports"}.
                     If None, predict_label() returns the numeric label instead.
        """
        self.data_x = data_x
        self.data_y = data_y
        self.data_label = data_label
        self.label_names = label_names

    def calculate_distance(self, point_x, point_y):
        return np.sqrt((self.data_x - point_x) ** 2 + (self.data_y - point_y) ** 2)

    def get_k_nearest_neighbors(self, point_x, point_y, k):
        distances = self.calculate_distance(point_x, point_y)
        nearest_indices = np.argsort(distances)[:k]
        return self.data_label[nearest_indices]

    def _predict_numeric_label(self, point_x, point_y, k):
        neighbors_labels = self.get_k_nearest_neighbors(point_x, point_y, k)
        unique_labels, counts = np.unique(neighbors_labels, return_counts=True)
        majority_label = unique_labels[np.argmax(counts)]
        return majority_label

    def predict_label(self, point_x, point_y, k):
        """Predicted class name if label_names was provided, otherwise the numeric label."""
        majority_label = self._predict_numeric_label(point_x, point_y, k)
        if self.label_names is not None:
            return self.label_names.get(majority_label, majority_label)
        return majority_label


knn = KNNCalculate(data_x, data_y, data_label, label_names=label_names)
print(knn.predict_label(test_point_x, test_point_y, k))

## 4. Worked Example

Predicting the class of test point **(9, 13)** — 9 hours streamed/week, 13 hours competing/week — with **k = 5**, using the full dataset above.

The cell below builds a pandas table of every account sorted by distance to the test point, so you can see exactly which ones get picked as the "nearest neighbors" (`is_neighbor`) and which don't.

In [ ]:
order = np.argsort(distances)

distance_df = pd.DataFrame({
    "x": data_x[order],
    "y": data_y[order],
    "label_name": [label_names[lbl] for lbl in data_label[order]],
    "distance": np.round(distances[order], 3),
}, index=order)
distance_df.index.name = "point"
distance_df.insert(0, "rank", range(1, len(distance_df) + 1))
distance_df["is_neighbor"] = distance_df["rank"] <= k

distance_df

With `k = 5`, the 5 nearest neighbors are two **Gamer** accounts (very close, distance ≈ 3.2 and 4.0), then two **E-Sports** accounts (distance ≈ 6.1 and 6.4), then one more **Gamer** account (distance ≈ 6.7).

Vote count: `Gamer` → 3 votes, `E-Sports` → 2 votes → **Gamer wins**, no tie needed this time — a genuine majority, unlike the original 4-point example where 3 classes tied 1-1-1.

Running the prediction confirms it:
```text
Predicted label for point (9, 13) with k=5: Gamer
```

Try changing `k` in the cells above (e.g. `k = 2`, which would only see the two closest Gamer neighbors) to see the vote — and the prediction — change.

## 5. Visualizing the Decision Boundary

This is the most useful plot for *understanding* KNN — the notebook equivalent of `Plot.plot_decision_boundary()`:

- It builds a grid covering the whole 2D space and asks the KNN model to predict a label for **every** grid cell → this produces the colored background regions (the "decision boundary"). Wherever the background color changes, that's where KNN's prediction flips from one class to another.
- The training points are drawn as a scatter on top, so you can see how the boundary follows the data.
- Passing a `test_point` also draws the test point as a **yellow star**, **dashed lines** to its `k` nearest neighbors, **black circles** highlighting those neighbors, and a dotted **search-radius circle** (radius = distance to the farthest of the k neighbors) showing the actual search area KNN used.

In [ ]:
def plot_decision_boundary(knn, k, x_range=None, y_range=None, resolution=150,
                            test_point=None, xlabel='X', ylabel='Y'):
    if x_range is None:
        x_min, x_max = knn.data_x.min() - 2, knn.data_x.max() + 2
    else:
        x_min, x_max = x_range

    if y_range is None:
        y_min, y_max = knn.data_y.min() - 2, knn.data_y.max() + 2
    else:
        y_min, y_max = y_range

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                          np.linspace(y_min, y_max, resolution))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    # _predict_numeric_label (not predict_label) because contourf needs numbers, not strings
    predictions = np.array([knn._predict_numeric_label(x, y, k) for x, y in grid_points])
    predictions = predictions.reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))
    contour = ax.contourf(xx, yy, predictions, alpha=0.3)
    ax.scatter(knn.data_x, knn.data_y, c=knn.data_label, edgecolor='k', cmap=plt.cm.coolwarm, zorder=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f'KNN Decision Boundary (k={k})')

    # KNN uses Euclidean distance (a real circle) -- lock the aspect ratio to 'equal' or the
    # search-radius circle drawn below would render as a misleading ellipse.
    ax.set_aspect('equal', adjustable='box')

    if knn.label_names is not None:
        unique_labels = sorted(knn.label_names.keys())
        cbar = fig.colorbar(contour, ax=ax, ticks=unique_labels)
        cbar.ax.set_yticklabels([knn.label_names[label] for label in unique_labels])

    if test_point is not None:
        test_x, test_y = test_point
        distances = knn.calculate_distance(test_x, test_y)
        nearest_indices = np.argsort(distances)[:k]

        # radius of the K-neighborhood = distance to the farthest of the k nearest neighbors
        k_radius = distances[nearest_indices].max()
        search_circle = Circle((test_x, test_y), k_radius, fill=False,
                                edgecolor='black', linestyle=':', linewidth=2,
                                zorder=2, label=f'Search radius (k={k}) = {k_radius:.2f}')
        ax.add_patch(search_circle)

        for idx in nearest_indices:
            ax.plot([test_x, knn.data_x[idx]], [test_y, knn.data_y[idx]],
                    color='black', linestyle='--', linewidth=1, zorder=2)

        ax.scatter(knn.data_x[nearest_indices], knn.data_y[nearest_indices],
                   s=250, facecolors='none', edgecolors='black', linewidths=2,
                   label=f'{k} Nearest Neighbors', zorder=4)

        predicted = knn.predict_label(test_x, test_y, k)
        ax.scatter(test_x, test_y, marker='*', s=350, c='yellow', edgecolors='black',
                   linewidths=1.5, label=f'Test point -> {predicted}', zorder=5)

        ax.set_xlim(min(x_min, test_x - k_radius * 1.1), max(x_max, test_x + k_radius * 1.1))
        ax.set_ylim(min(y_min, test_y - k_radius * 1.1), max(y_max, test_y + k_radius * 1.1))
        ax.legend(loc='best')

    plt.show()


plot_decision_boundary(knn, k, test_point=(test_point_x, test_point_y), xlabel=xlabel, ylabel=ylabel)

predicted_label = knn.predict_label(test_point_x, test_point_y, k)
print(f"Predicted label for point ({test_point_x}, {test_point_y}) with k={k}: {predicted_label}")

## 6. Understanding the Plots

**The raw scatter plot** is just "what the data looks like" before any prediction happens — you can already see the three clusters forming: high-streamed/low-competing (Streamer), low/low (Gamer), and low-streamed/high-competing (E-Sports).

**The decision boundary plot** turns the abstract "distance + vote" math into something you can see directly:

- The colored regions show what class KNN would predict for *any* point in that area, given the current `k`.
- The star, the dashed lines, and the circled neighbors *are* the algorithm — literally the k accounts that got to "vote" on the test account's class, and the boundary that their vote produced.
- Our test account (9 streamed h/wk, 13 competing h/wk) sits right where the Gamer and E-Sports regions meet, which is exactly why its 5 nearest neighbors are a mix of both classes rather than a landslide for one.

## 7. Things to Try (Exercises)

All of these are edits to the cells above — no other files need to change:

1. **Change `k`** in Section 3/5 (try `1`, `2`, `9`) and re-run the decision boundary cell. Small `k` → boundary follows individual points closely (can overfit / be noisy). Large `k` → boundary gets smoother but may ignore local structure.
2. **Add more accounts** to `data_x` / `data_y` / `data_label` in Section 2 and re-run everything below — watch the decision boundary regions reshape.
3. **Move the test point** (`test_point_x`, `test_point_y`) around and watch which neighbors light up and how the predicted class changes.
4. **Force a tie** on purpose (e.g. pick a test point equidistant from one account of each class) and confirm which label wins, to understand the tie-breaking rule from Step 3.
5. **Swap in a different real classification problem** you care about (e.g. billing tiers by usage minutes vs. monthly spend, or customer segments by two behavioral metrics) by replacing the dataset in Section 2 and the `xlabel`/`ylabel`/`label_names`.
6. **Compare against the config-driven CLI version** — `main.py` + `config_dev.yaml` in this repository run the same algorithm on a different (smaller) dataset, driven by a YAML file instead of notebook cells.

## 8. Limitations (by design, for learning)

This implementation is intentionally minimal so the algorithm stays readable — it does **not** include things a production KNN would need:

- No feature scaling/normalization — since `x` and `y` are on similar-ish scales here, this doesn't matter much for the demo, but with real, differently-scaled features you'd normally standardize features first (distance would otherwise be dominated by whichever feature has the larger numeric range).
- Only works with 2 features (`x`, `y`) — real KNN generalizes to any number of features/dimensions.
- Only Euclidean distance is implemented (other options include Manhattan or cosine distance).
- `k` is set manually rather than tuned automatically (e.g. via cross-validation).
- The dataset here, while more realistic than four arbitrary points, is still small and hand-authored rather than measured from a real streaming platform — it's built to make the clusters and the decision boundary easy to see and reason about, not to be a rigorous sample of real user behavior.

These are great follow-up topics once you're comfortable with the basic distance + majority-vote mechanics shown here.